## Import Modules

In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.ensemble import GradientBoostingRegressor

## Load Dataset

In [2]:
df = pd.read_csv('Housing.csv')

df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB


,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


## Data preprocessing and Feature Encoding

In [3]:
df = pd.get_dummies(df, drop_first=True)

In [4]:
df.isnull().sum()

price                              0
area                               0
bedrooms                           0
bathrooms                          0
stories                            0
parking                            0
mainroad_yes                       0
guestroom_yes                      0
basement_yes                       0
hotwaterheating_yes                0
airconditioning_yes                0
prefarea_yes                       0
furnishingstatus_semi-furnished    0
furnishingstatus_unfurnished       0
dtype: int64

## Feature selection and train/test split

In [5]:
X = df.drop('price', axis=1)

y = df['price']

In [6]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=0.2,
    random_state=42
)

## Linear Regression - Baseline Model

In [8]:
model_lr = LinearRegression()

model_lr.fit(X_train, y_train)

LinearRegression()

## Prediction

In [9]:
y_pred_lr = model_lr.predict(X_test)

## Evaluation

In [22]:
def evaluate(y_test, y_pred_lr):

    mae_lr = mean_absolute_error(y_test, y_pred_lr)

    rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

    r2_lr = r2_score(y_test, y_pred_lr)

    return mae_lr, rmse_lr, r2_lr

mae_lr, rmse_lr, r2_lr = evaluate(y_test, y_pred_lr)

## Decision Tree Model 

In [11]:
dt = DecisionTreeRegressor(random_state=42)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

## Evaluation 

In [24]:
def evaluate(y_test, y_pred_dt):

    mae_dt = mean_absolute_error(y_test, y_pred_dt)

    rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

    r2_dt = r2_score(y_test, y_pred_dt)

    return mae_dt, rmse_dt, r2_dt

mae_dt, rmse_dt, r2_dt = evaluate(y_test, y_pred_dt)

## Random Forest

In [13]:
rf = RandomForestRegressor(random_state=42)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

## Evaluation 

In [25]:
def evaluate(y_test, y_pred_rf):

    mae_rf = mean_absolute_error(y_test, y_pred_rf)

    rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

    r2_rf = r2_score(y_test, y_pred_rf)

    return mae_rf, rmse_rf, r2_rf

mae_rf, rmse_rf, r2_rf = evaluate(y_test, y_pred_rf)

## GradientBoosting

In [30]:
gb = GradientBoostingRegressor(random_state=42)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

## Evaluation 

In [31]:
def evaluate(y_test, y_pred_gb):

    mae_gb = mean_absolute_error(y_test, y_pred_gb)

    rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

    r2_gb = r2_score(y_test, y_pred_gb)

    return mae_gb, rmse_gb, r2_gb

mae_gb, rmse_gb, r2_gb = evaluate(y_test, y_pred_gb)

## Hyperparameter Tuning

In [32]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2'
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
             param_grid={'max_depth': [5, 10, None],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='r2')

## Best Model

In [36]:
best_rf = grid.best_estimator_

y_pred_tuned = best_rf.predict(X_test)

## Evaluation 

In [37]:
def evaluate(y_test, y_pred_tuned):

    mae_tuned = mean_absolute_error(y_test, y_pred_tuned)

    rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))

    r2_tuned = r2_score(y_test, y_pred_tuned)

    return mae_tuned, rmse_tuned, r2_tuned

mae_tuned, rmse_tuned, r2_tuned = evaluate(y_test, y_pred_tuned)

## Comparison Table

In [41]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'Decision Tree',
        'Random Forest',
        'Gradient Boosting',
        'Tuned Random Forest'
    ],
    'R2 Score': [
        r2_lr,
        r2_dt,
        r2_rf,
        r2_gb,
        r2_tuned
    ],
    'MAE Score': [
        mae_lr,
        mae_dt,
        mae_rf,
        mae_gb,
        mae_tuned
    ],
    'RMSE Score': [
        rmse_lr,
        rmse_dt,
        rmse_rf,
        rmse_gb,
        rmse_tuned
    ]
})

print(results)

                 Model  R2 Score     MAE Score    RMSE Score
0    Linear Regression  0.652924  9.700434e+05  1.324507e+06
1        Decision Tree  0.477146  1.195266e+06  1.625670e+06
2        Random Forest  0.611919  1.021546e+06  1.400566e+06
3    Gradient Boosting  0.665965  9.597490e+05  1.299386e+06
4  Tuned Random Forest  0.600387  1.035775e+06  1.421223e+06
